# 9주차 랩 · 책 동아리 고객에게 함께 권할 분야

**⏱ 60분** · Charles Book Club 고객 4,000명 · 개인 제출

## 🎯 이번 랩의 질문

교재에서는 식료품점 영수증으로 함께 권할 품목을 찾았습니다. 이번에는 책 동아리 고객 4,000명의 분야별 구매 기록으로 같은 일을 합니다. 책 동아리는 고객에게 다음에 살 책을 우편으로 권합니다.

> **어떤 분야의 책을 산 고객에게 어떤 분야를 함께 권할까요?**

교재에서 반복한 패턴을 책 동아리 자료에 다시 씁니다.

| 패턴 | 교재 | 이번 랩 |
|---|---|---|
| 참·거짓 표 만들기 | 1절 | 과제 1 |
| 열의 평균으로 지지도 | 2절 | 과제 2 |
| 규칙 하나의 신뢰도·향상도 직접 계산 | 2절 | 과제 3 |
| `apriori` → `association_rules` → 개수 | 3절 | 과제 4 |
| antecedent가 하나인 규칙을 향상도로 읽기 | 4절 | 과제 5 |
| 기준을 바꿔 규칙 수 보기 | 📝 과제 2 | 과제 6 |
| 권할 규칙 하나 고르기 | 5절 | 과제 7 |

### 진행 방법

| 표시 | 뜻 | 할 일 |
|---|---|---|
| ▶ | 제공 코드 | 그대로 실행합니다 |
| 📝 | 과제 | `None` 자리를 코드로 바꿔 채웁니다. 변수 이름은 바꾸지 않습니다 |
| ✅ | 확인 셀 | 고치지 말고 실행만 합니다 |
| 💬 | 결과 읽기 | 결과에서 확인할 숫자를 짚습니다 |

- 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다.
- 확인 셀은 변수의 형식만 봅니다. "형식 확인 완료"는 값이 맞다는 뜻이 아닙니다. 값은 제출 후 채점합니다.

## ▶ 준비

책 동아리 Charles Book Club의 고객 4,000명 기록을 읽습니다(Shmueli 외, *Machine Learning for Business Analytics* 2판의 사례 자료, `mlba` 패키지). **한 행은 고객 한 명**이고, 책 분야 11열에는 그 분야의 책을 산 권수가 들어 있습니다.

| 열 | 분야 | 열 | 분야 |
|---|---|---|---|
| `ChildBks` | 아동 | `GeogBks` | 지리 |
| `YouthBks` | 청소년 | `ItalCook` | 이탈리아 요리 |
| `CookBks` | 요리 | `ItalAtlas` | 이탈리아 지도 |
| `DoItYBks` | DIY | `ItalArt` | 이탈리아 미술 |
| `RefBks` | 참고서 | `Florence` | 피렌체 미술 |
| `ArtBks` | 미술 | | |

준비 셀이 만드는 것:
- `df` — 고객 4,000명의 기록(24열)
- `books` — 책 분야 11열의 이름
- `show_rules(rules)` — 교재의 같은 이름 함수입니다. 규칙 표를 글자로 바꿔 향상도 순으로 보여 줍니다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. 책 동아리 구매 기록을 읽습니다.
%pip install -q mlba "mlxtend==0.23.4"
import pandas as pd
pd.set_option("display.max_rows", 500)       # 표를 줄임표(...) 없이 끝까지 보여 줌
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", None)
import mlba
from mlxtend.frequent_patterns import apriori, association_rules
import os
import sys

for folder in (".", "..", "../..", "../../.."):
    if os.path.exists(os.path.join(folder, "balab.py")):
        sys.path.insert(0, folder)
        break
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py", "balab.py")

from balab import checker, summary, check_format
check = checker("week09")

df = mlba.load_data("CharlesBookClub.csv")
books = ["ChildBks", "YouthBks", "CookBks", "DoItYBks", "RefBks", "ArtBks",
         "GeogBks", "ItalCook", "ItalAtlas", "ItalArt", "Florence"]

def show_rules(rules):
    table = rules[["antecedents", "consequents", "support", "confidence", "lift"]].copy()
    table["antecedents"] = table["antecedents"].apply(lambda items: ", ".join(sorted(items)))
    table["consequents"] = table["consequents"].apply(lambda items: ", ".join(sorted(items)))
    return table.sort_values("lift", ascending=False, kind="stable").round(3)


print("준비 완료!", df.shape)
df[books].head()

✅ `준비 완료! (4000, 24)`가 나오면 됩니다. 첫 행 고객은 `YouthBks`(청소년)와 `CookBks`(요리)를 한 권씩 샀습니다.

---
## 📝 과제 1 · 고객 × 분야 참·거짓 표

교재 1절처럼 규칙을 찾을 참·거짓 표를 만듭니다. 이 자료의 칸은 산 권수이므로, 한 권 이상 샀으면 참으로 바꿉니다.

- `df`에서 `books` 열만 골라 `> 0`을 붙여 `X`에 담습니다.
- `X`의 행 수(고객 수)를 `n_customers`, 열 수(분야 수)를 `n_books`에 담습니다.

**결과 형식:** `n_customers`, `n_books`는 정수

In [ ]:
# 📝 과제 1 — None 자리를 채우세요
# 힌트: 열 이름 목록으로 열을 고른 뒤 > 0을 붙입니다. 행·열 수는 shape에 있습니다.
X = None            # 고객 × 분야 참·거짓 표
n_customers = None  # 고객 수
n_books = None      # 분야 수

In [ ]:
# ✅ 확인 — 과제 1 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(1, n_customers=n_customers, n_books=n_books)
else:
    check_format(1, n_customers=n_customers, n_books=n_books)

💬 **결과 읽기** — 고객과 분야는 각각 몇 개인가요?

---
## 📝 과제 2 · 지지도가 높은 분야 셋

교재 2절처럼 열의 평균으로 분야마다 지지도를 구합니다.

- 지지도를 큰 순서로 정렬해 상위 3개를 `top_support`에 담습니다. 인덱스는 분야 이름입니다.

**결과 형식:** `top_support`는 숫자 Series 3개(인덱스: 분야 이름)

In [ ]:
# 📝 과제 2 — None 자리를 채우세요
# 힌트: 교재 2절의 첫 셀과 같은 한 줄에서 head의 개수만 바꿉니다.
top_support = None  # 지지도 상위 3개 분야

In [ ]:
# ✅ 확인 — 과제 2 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(2, top_support=(top_support, 0.001))
else:
    check_format(2, top_support=top_support)

💬 **결과 읽기** — 1위 분야는 고객 100명 중 몇 명꼴로 샀나요?

---
## 📝 과제 3 · 규칙 하나 직접 계산

교재 2절처럼 규칙 **{RefBks} ⇒ {CookBks}**(참고서를 산 고객은 요리책도 산다)의 신뢰도와 향상도를 계산합니다.

- 두 분야를 함께 산 고객을 `both`에 표시합니다.
- 신뢰도를 `confidence`, 향상도를 `lift`에 담습니다.

**결과 형식:** `confidence`, `lift`는 숫자

In [ ]:
# 📝 과제 3 — None 자리를 채우세요
# 힌트: 교재 2절의 두 셀에서 표 이름을 X로, 품목 이름을 분야 이름으로 바꿉니다.
both = None        # 참고서와 요리책을 함께 산 고객
confidence = None  # 신뢰도
lift = None        # 향상도

In [ ]:
# ✅ 확인 — 과제 3 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(3, confidence=(confidence, 0.001), lift=(lift, 0.001))
else:
    check_format(3, confidence=confidence, lift=lift)

💬 **결과 읽기** — 참고서를 산 고객 중 요리책도 산 고객의 비율은 얼마이고, 향상도는 1보다 큰가요?

---
## 📝 과제 4 · frequent itemset과 규칙

교재 3절의 두 줄을 `X`에 씁니다. 지지도 기준은 **0.05**(고객 200명 이상), 신뢰도 기준은 **0.5**입니다.

- frequent itemset을 `itemsets`, 규칙을 `rules`에 담습니다.
- 각각의 행 수를 `n_itemsets`, `n_rules`에 담습니다.

**결과 형식:** `n_itemsets`, `n_rules`는 정수

In [ ]:
# 📝 과제 4 — None 자리를 채우세요
# 힌트: 교재 3절의 apriori 한 줄과 association_rules 한 줄에서 기준값만 바꿉니다. 행 수는 len()입니다.
itemsets = None    # 지지도 0.05 이상인 frequent itemset
rules = None       # 신뢰도 0.5 이상인 규칙
n_itemsets = None  # frequent itemset 수
n_rules = None     # 규칙 수

In [ ]:
# ✅ 확인 — 과제 4 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(4, n_itemsets=n_itemsets, n_rules=n_rules)
else:
    check_format(4, n_itemsets=n_itemsets, n_rules=n_rules)

💬 **결과 읽기** — frequent itemset과 규칙은 각각 몇 개인가요?

---
## 📝 과제 5 · antecedent가 한 분야인 규칙

"이 분야를 산 고객에게 무엇을 권할까"에 바로 답하는 것은 antecedent가 한 분야인 규칙입니다(교재 4절).

- `rules`에서 antecedent가 한 분야인 규칙만 `rules_one`에 담습니다.
- 그 개수를 `n_one`, 향상도의 최댓값을 `best_lift`에 담습니다.
- `show_rules(rules_one).head(10)`으로 상위 규칙을 읽습니다.

**결과 형식:** `n_one`은 정수, `best_lift`는 숫자

In [ ]:
# 📝 과제 5 — None 자리를 채우세요
# 힌트: 교재 4절의 조건(antecedents에 apply(len) == 1)으로 행을 고릅니다. 최댓값은 lift 열의 max()입니다.
rules_one = None  # antecedent가 한 분야인 규칙
n_one = None      # 그 규칙의 수
best_lift = None  # 그 규칙 가운데 가장 큰 향상도

In [ ]:
# ✅ 확인 — 과제 5 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(5, n_one=n_one, best_lift=(best_lift, 0.01))
else:
    check_format(5, n_one=n_one, best_lift=best_lift)

💬 **결과 읽기** — 향상도 1위 규칙의 antecedent와 consequent는 어떤 분야인가요?

---
## 📝 과제 6 · 지지도 기준을 0.025로 낮추면

과제 4의 두 줄에서 지지도 기준만 **0.025**(고객 100명 이상)로 바꿉니다. 신뢰도 기준은 0.5 그대로입니다.

- frequent itemset을 `itemsets_low`, 규칙을 `rules_low`에 담고, 규칙 수를 `n_rules_low`에 담습니다.

**결과 형식:** `n_rules_low`는 정수

In [ ]:
# 📝 과제 6 — None 자리를 채우세요
# 힌트: 과제 4의 apriori·association_rules 두 줄을 그대로 쓰고 min_support만 바꿉니다.
itemsets_low = None  # 지지도 0.025 이상인 frequent itemset
rules_low = None     # 그 itemset의 규칙
n_rules_low = None   # 규칙 수

In [ ]:
# ✅ 확인 — 과제 6 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(6, n_rules_low=n_rules_low)
else:
    check_format(6, n_rules_low=n_rules_low)

💬 **결과 읽기** — 규칙 수는 과제 4의 몇 배가 되었나요?

---
## 📝 과제 7 · 본인의 판단

1. 참고서(`RefBks`)를 산 고객에게 우편으로 권할 분야 하나를 과제 5의 표에서 고르고, 그 규칙의 신뢰도와 향상도를 적으세요.
2. 지지도 기준 0.05와 0.025 가운데 무엇을 쓸지 한 문장으로 적으세요.

자동 채점은 하지 않습니다.

✏️ **나의 답:** (여기를 두 번 눌러 적습니다)

---
## ✅ 제출 전 확인

아래 셀을 실행해 과제 1~6의 작성 여부와 형식을 확인합니다. "형식 확인 완료"는 계산값이 맞다는 뜻이 아닙니다. 결과를 직접 읽고 과제 7을 작성해 제출합니다. 실제 값 채점은 제출 후 진행합니다.

In [ ]:
# ✅ 제출 전 확인 — 과제 1~6의 작성 여부와 형식
summary()